# 02. Вуглець, CO₂ і кисень: від карти біомаси до показників екосистемних послуг

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yevheniikhan/kyiv-forest-agentic-ai/blob/main/notebooks/02_carbon_oxygen_services.ipynb)

Записник переводить надземну біомасу в запас вуглецю, еквівалент CO₂ і стехіометричний еквівалент кисню,
порівнює результат із грідованим продуктом GEDI L4B та з регіональними значеннями для Київської області
(база даних «Укрдержліспроекту» станом на 01.01.2011).

Обмеження: карта охоплює лише надземну деревну біомасу. Корені, підлісок, мертва деревина і ґрунт не враховані,
тому загальний запас вуглецю буде більшим.

In [ ]:
# Код репозиторію і бібліотеки
import os, sys
if not os.path.exists('/content/kyiv-forest-agentic-ai'):
    !git clone -q https://github.com/yevheniikhan/kyiv-forest-agentic-ai /content/kyiv-forest-agentic-ai
sys.path.insert(0, '/content/kyiv-forest-agentic-ai/src')
!pip -q install geemap pyproj

import ee
PROJECT = 'gen-lang-client-0200925587'   # замініть на свій проєкт Google Cloud з увімкненим Earth Engine
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()                     # вікно авторизації підтверджує користувач
    ee.Initialize(project=PROJECT)
print('Earth Engine:', ee.String('готово').getInfo())

In [ ]:
from kyivforest import config as C, ee_data as D, services as S
import pandas as pd
fc = D.training_samples(max_n=4000)
bands = D.predictors().bandNames().getInfo()
agb = D.ee_rf_map(fc, bands)
st = D.area_stats(agb).getInfo()
forest_ha, agb_tot = st['ha_sum'], st['agb_tot_sum']
pools = S.biomass_to_pools(agb_tot)
l4b = ee.Image('LARSE/GEDI/GEDI04_B_002').select('MU').updateMask(D.forest_mask())
l4b_mean = l4b.reduceRegion(ee.Reducer.mean(), D.aoi(), 1000).getInfo()['MU']
tab = pd.DataFrame({
    'показник': ['площа лісу, га', 'середня AGB (RF), Мг/га', 'середня AGBD GEDI L4B, Мг/га',
                 'надземна біомаса, тис. т', 'вуглець, тис. т C', 'еквівалент CO₂, тис. т',
                 'частка від регіональної щільності фітомаси 2011 р.'],
    'значення': [forest_ha, agb_tot / forest_ha, l4b_mean, pools['agb_Mg'] / 1e3,
                 pools['carbon_Mg'] / 1e3, pools['co2_Mg'] / 1e3, S.compare_with_region(agb_tot / forest_ha)]})
tab['значення'] = tab['значення'].round(2)
tab.to_csv('services_summary.csv', index=False); tab

## Регіональний контекст (Київська область, 2011 р.)

| Показник | Значення |
|---|---|
| Площа, вкрита лісовою рослинністю | 655,4 тис. га |
| Фітомаса | 111,0 млн т (180 т/га) |
| Вуглець рослинної біомаси | 61,8 млн т |
| Щорічне депонування вуглецю | 3,15 млн т C/рік |
| Продукування кисню | 9,13 млн т O₂/рік |

Карта з космічних даних дає просторовий розподіл, а таксаційні дані дають повні запаси за фракціями. Для звітності їх поєднують, а не замінюють одне одним.

## Як це пов'язати з Living Lab LandShift у Київській області

- моніторинг змін землекористування за відкритими даними: щорічне оновлення карти біомаси і втрат лісу;
- зелена інфраструктура й агролісомеліорація: оцінка вуглецю в полезахисних смугах і нових насадженнях;
- водно-болотні угіддя: маски і тренди за Sentinel-1/2;
- воркшопи з громадами: карти й таблиці у форматах, які місцеві фахівці можуть перевірити самостійно.